In [1]:
# loading libraries

import numpy as np # linear algebra
import pandas as pd # data processing
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

Let's have a look at the layoffs in the tech world up to 2026 in comparison to the years before.

First step: loading the data

In [9]:
data = pd.read_csv("/Users/ulrike_imac_air/projects/DataScienceProjects/tech_layoffs/tech_layoffs_csv/tech_layoffs_til_2026.csv", delimiter=",")

print(data.columns)

In [10]:
summary = pd.DataFrame({
    'Non-NA Count': data.count(),
    'NA Count': data.isna().sum()
})
print(summary)


In [11]:
# Drop rows with NaN in 'Laid_Off' column
data_cleaned = data.dropna(subset=['Laid_Off'])

# Verify the cleaned data
print(data_cleaned.shape)  # Check rows and columns

# Count non-NA values in the cleaned data
non_na_counts = data_cleaned.count()
print(non_na_counts)


In [12]:
sum_laid_off = data_cleaned['Laid_Off'].sum()
print(sum_laid_off)

# Group by 'Year' and sum 'Laid_Off'
sum_laid_off_per_year = data_cleaned.groupby('Year')['Laid_Off'].sum().reset_index()
print(sum_laid_off_per_year)

In [13]:
# Create a bar chart using Seaborn
plt.figure(figsize=(10, 6))
sns.barplot(data=sum_laid_off_per_year, x='Year', y='Laid_Off', color='darkblue')

# Add titles and labels
plt.title('Sum of Laid Off Per Year')
plt.xlabel('Year')
plt.ylabel('Total Laid Off')

# Show the plot
plt.show()

In [ ]:
# Interactive Plotly bar chart showing layoffs per year
fig_bar = px.bar(sum_laid_off_per_year, x='Year', y='Laid_Off', 
                 title='Interactive: Sum of Laid Off Per Year', 
                 labels={'Laid_Off': 'Total Laid Off'},
                 color='Laid_Off', color_continuous_scale='Viridis')
fig_bar.show()

Layoffs in tech remained high in 2024 and through to 2026, as seen in the visualizations.

In [14]:
# Group by 'Year' and 'Country' and sum 'Laid_Off'
sum_laid_off_per_year_country = data_cleaned.groupby(['Year', 'Country'])['Laid_Off'].sum().reset_index()

print(sum_laid_off_per_year_country.head())

In [15]:
# Group by 'Year' and 'Country' and sum 'Laid_Off'
sum_laid_off_per_year_country = data_cleaned.groupby(['Year', 'Country'])['Laid_Off'].sum().reset_index()

# Get unique years
years = sum_laid_off_per_year_country['Year'].unique()

# Create a pie chart for each year
for year in years:
    # Filter data for the specific year
    year_data = sum_laid_off_per_year_country[sum_laid_off_per_year_country['Year'] == year]
    
    # Sort the data by 'Laid_Off' and get top 5 countries
    top_countries = year_data.nlargest(5, 'Laid_Off')
    
    # Get the remaining countries and sum their 'Laid_Off' values into "Others"
    others_sum = year_data[~year_data['Country'].isin(top_countries['Country'])]['Laid_Off'].sum()
    others = pd.DataFrame({'Country': ['Others'], 'Laid_Off': [others_sum]})
    
    # Combine top countries with "Others"
    final_data = pd.concat([top_countries, others])
    
    # Use Seaborn 'Set1' color palette
    colors = sns.color_palette('Set2', len(final_data))
    
    # Define 'explode' to slightly separate the largest slices (e.g., the top 5 countries)
    explode = [0.1 if country != 'Others' else 0 for country in final_data['Country']]

    # Plot the pie chart
    plt.figure(figsize=(7, 7))
    wedges, texts, autotexts = plt.pie(final_data['Laid_Off'], labels=final_data['Country'], autopct='%1.1f%%', 
                                       startangle=45, colors=colors, explode=explode, labeldistance=1.1)

    # Adjust label font size
    for text in texts + autotexts:
        text.set_fontsize(9)

    plt.title(f'Laid Off Distribution in {year}')
    plt.axis('equal')  # Equal aspect ratio ensures the pie is drawn as a circle.
    
    # Ensure the layout is adjusted so labels don't overlap
    plt.tight_layout()
    plt.show()

In [ ]:
# Interactive Plotly sunburst chart: Continent -> Country -> Industry
fig_sunburst = px.sunburst(data_cleaned, 
                           path=['Continent', 'Country', 'Industry'], 
                           values='Laid_Off',
                           title='Layoffs Hierarchy: Continent -> Country -> Industry',
                           color='Continent')
fig_sunburst.show()

In [ ]:
# Interactive Plotly scatter_geo map showing layoffs globally
# Grouping by Location to avoid overlapping bubbles in the animation
data_geo = data_cleaned.groupby(['Year', 'Country', 'latitude', 'longitude', 'Industry'])['Laid_Off'].sum().reset_index()

fig_geo = px.scatter_geo(data_geo, 
                         lat='latitude', 
                         lon='longitude', 
                         color='Country', 
                         hover_name='Country', 
                         size='Laid_Off', 
                         animation_frame='Year',
                         title='Global Tech Layoffs Over Time (2020-2026)',
                         projection='natural earth')
fig_geo.show()